In [5]:
import pandas as pd
import numpy as np

# load data
df = pd.read_excel('Copy of PEMBELIAN MEY NOVEMBEREV.xlsx')

df.head()


,invoiceno,invoicedate,suppid,grandtotal,totdischd,taxpercent,tottax,totnetto,note,linesno,...,disc,totdisc,description,description_barang,unitid,description.1,years8,disc2,disc3,disc4
0,FBT2505-00001,2025-05-01,ESB,240000,0,0,0,240000,NaN,1,...,0,0,PT. EMPATI SUKSES BERKARYA,JOINT STEER GRANDMAX,PCS,PT ANUGRAH SAMARINDA,20250501,0,0,0
1,FBT2505-00002,2025-05-01,JAM,1060000,0,0,0,1060000,NaN,1,...,0,0,JAYA ABADI MOTOR,ACCU INCOE N70,PCS,PT ANUGRAH SAMARINDA,20250501,0,0,0
2,FBT2505-00005,2025-05-03,ESB,7260000,0,0,0,7260000,NaN,1,...,0,0,PT. EMPATI SUKSES BERKARYA,FASTRON TECHNO 4LTR,GLN,PT ANUGRAH SAMARINDA,20250503,0,0,0
3,FBT2505-00005,2025-05-03,ESB,7260000,0,0,0,7260000,NaN,2,...,0,0,PT. EMPATI SUKSES BERKARYA,COOLANT RADIATOR,PCS,PT ANUGRAH SAMARINDA,20250503,0,0,0
4,FBT2505-00005,2025-05-03,ESB,7260000,0,0,0,7260000,NaN,3,...,0,0,PT. EMPATI SUKSES BERKARYA,RACK STEER ASSY GRANDMAX,PCS,PT ANUGRAH SAMARINDA,20250503,0,0,0


In [7]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

df['invoicedate'] = pd.to_datetime(df['invoicedate'])
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df.dropna(subset=['description_barang', 'description', 'amount'], inplace=True)


In [10]:
harga_termurah = df.groupby(['description_barang','description'])['amount'].min().reset_index()

In [11]:
recency = df.groupby(['description_barang','description'])['invoicedate'].max().reset_index()
recency['recency_days'] = (recency['invoicedate'].max() - recency['invoicedate']).dt.days


In [14]:
freq = df.groupby(['description_barang','description']).size().reset_index(name='frequency')

In [15]:
model_df = harga_termurah \
    .merge(recency[['description_barang','description','recency_days']], on=['description_barang','description']) \
    .merge(freq, on=['description_barang','description'])

model_df.head()

,description_barang,description,amount,recency_days,frequency
0,ACCU FB PREMIUM 35A,PT. EMPATI SUKSES BERKARYA,641000,24,1
1,ACCU GS N70,JAYA ABADI MOTOR,1190000,102,1
2,ACCU INCOE N70,JAYA ABADI MOTOR,1015000,141,2
3,ACCU YUASA MF NS402L 35A,JAYA ABADI MOTOR,670000,63,1
4,ACCU ZUUR YUASA,METRO MOTOR,80000,50,1


In [17]:
# normalisasi
model_df['score_price'] = 1 / model_df['amount']
model_df['score_recency'] = 1 / (1 + model_df['recency_days'])
model_df['score_freq'] = model_df['frequency'] / model_df['frequency'].max()

# skor akhir
model_df['final_score'] = (
    model_df['score_price'] * 0.5 +
    model_df['score_recency'] * 0.3 +
    model_df['score_freq'] * 0.2
)

In [19]:
def rekomendasi_supplier(nama_barang, top_n=5):
    hasil = model_df[model_df['description_barang'].str.contains(nama_barang, case=False, na=False)]
    hasil = hasil.sort_values('final_score', ascending=False)
    return hasil[['description','amount','recency_days','frequency','final_score']]

In [20]:
rekomendasi_supplier("Baut")

,description,amount,recency_days,frequency,final_score
44,UD SINAR BAUT,20000,117,2,0.047012
37,UD SINAR BAUT,48000,46,1,0.028616
36,MITRA BAUT,10000,50,1,0.028155
38,76 MOTOR,20000,61,1,0.027086
40,76 MOTOR,30000,61,1,0.027078
45,UD SINAR BAUT,25000,75,1,0.026190
32,MITRA BAUT,10000,85,1,0.025761
41,76 MOTOR,80000,86,1,0.025677
386,76 MOTOR,60000,117,1,0.024773
39,JAYA ABADI MOTOR,160000,122,1,0.024664


In [21]:
model_df.to_csv("model_supplier.csv", index=False)